In [1]:
! pip install tensorflow


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
**Import Libaray**

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
import joblib
import datetime


load data

In [20]:
import json

with open('clean_recipes_5000.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    print("Jumlah resep:", len(data))

print("Struktur data awal:")
display(df.head())

Jumlah resep: 5000
Struktur data awal:


,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps,judul_bersih,bahan_bersih,Quality Score
0,Bakso Sapi (Pakai Blender),250 gram daging sapi--35 gram es batu (sekitar...,"1) Siapkan semua bahan, cincang daging sapi ke...",939,https://cookpad.com/id/resep/3431604-bakso-sap...,sapi,bakso sapi ( pakai blender ),8,"daging sapi , es batu cmx , putih telur , tepu...",10,bakso sapi pakai blender,250 gram daging sapi 35 gram es batu sekitar 4...,0.7997
1,Chilli Tuna Puff Kilat Super Yummy,1/2 pack Kulit puff instant saya merk.Edo--Bah...,1) Langkah \n1.Tumis chili tuna chunk dengan m...,516,https://cookpad.com/id/resep/4463240-chilli-tu...,ikan,chilli tuna puff kilat super yummy,9,"pack kulit puff instant merkedo , isian , kale...",3,chilli tuna puff kilat super yummy,1 2 pack kulit puff instant saya merk edo baha...,0.4689
2,Perkedel Tahu Simple,3 buah tahu petak--1 batang daun seledri--2 si...,1) Giling halus bawang merah+bawang putih+ mer...,481,https://cookpad.com/id/resep/4337985-perkedel-...,tahu,perkedel tahu simple,8,"tahu petak , batang daun seledri , bawang puti...",5,perkedel tahu simple,3 buah tahu petak 1 batang daun seledri 2 siun...,0.4827
3,Orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera--3 buah cab...,1) Goreng tempe yg sdh di potong2 dlm minyak p...,452,https://cookpad.com/id/resep/3989566-orek-temp...,tempe,orek tempe basah bumbu ulek,12,"papan tempe potong , cabe ijo , kecap manis , ...",3,orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera 3 buah cabe...,0.4223
4,Sop Iga Sapi Enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan--Secukupn...","1) Didihkan secukupnya air, lalu masukan poton...",375,https://cookpad.com/id/resep/3310336-sop-iga-s...,sapi,sop iga sapi enaaak bangeet,22,"iga sapi , tiriskan , air didihkan utk rebusan...",4,sop iga sapi enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan secukupny...",0.3550


In [21]:

df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Title                5000 non-null   str    
 1   Ingredients          5000 non-null   str    
 2   Steps                5000 non-null   str    
 3   Loves                5000 non-null   int64  
 4   URL                  5000 non-null   str    
 5   Category             5000 non-null   str    
 6   Title Cleaned        5000 non-null   str    
 7   Total Ingredients    5000 non-null   int64  
 8   Ingredients Cleaned  5000 non-null   str    
 9   Total Steps          5000 non-null   int64  
 10  judul_bersih         5000 non-null   str    
 11  bahan_bersih         5000 non-null   str    
 12  Quality Score        5000 non-null   float64
dtypes: float64(1), int64(3), str(9)
memory usage: 507.9 KB


Title                  0
Ingredients            0
Steps                  0
Loves                  0
URL                    0
Category               0
Title Cleaned          0
Total Ingredients      0
Ingredients Cleaned    0
Total Steps            0
judul_bersih           0
bahan_bersih           0
Quality Score          0
dtype: int64

In [22]:
df.describe()

,Loves,Total Ingredients,Total Steps,Quality Score
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,22.000000,12.507000,5.448800,0.300001
std,30.643339,4.615259,2.220172,0.068726
min,6.000000,3.000000,2.000000,0.084200
25%,9.000000,9.000000,4.000000,0.255600
50%,12.000000,12.000000,5.000000,0.293400
75%,25.000000,16.000000,7.000000,0.337675
max,939.000000,25.000000,23.000000,0.799700


In [23]:
import json
from collections import Counter

# Load data resep
with open('clean_recipes_5000.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Hitung distribusi kategori
categories = [item.get('Category', 'Unknown') for item in data]
category_counts = Counter(categories)

print("Distribusi Kategori:")
for cat, count in category_counts.items():
    print(f"{cat}: {count}")

# Cek apakah seimbang
total = len(data)
print(f"\nTotal resep: {total}")
print("Kategori mayoritas:", category_counts.most_common(1)[0][0], "dengan", category_counts.most_common(1)[0][1], "resep")
print("Kategori minoritas:", category_counts.most_common()[-1][0], "dengan", category_counts.most_common()[-1][1], "resep")

# Hitung rasio
majority = category_counts.most_common(1)[0][1]
minority = category_counts.most_common()[-1][1]
ratio = majority / minority
print(f"Rasio mayoritas/minoritas: {ratio:.2f}")

if ratio > 2:
    print("Dataset tidak seimbang, mungkin ada bias terhadap kategori mayoritas.")
else:
    print("Dataset cukup seimbang.")

Distribusi Kategori:
sapi: 625
ikan: 625
tahu: 625
tempe: 625
kambing: 625
telur: 625
ayam: 625
udang: 625

Total resep: 5000
Kategori mayoritas: sapi dengan 625 resep
Kategori minoritas: udang dengan 625 resep
Rasio mayoritas/minoritas: 1.00
Dataset cukup seimbang.


In [27]:
# Buat kolom ingredients_clean (tanpa cleaning lanjutan, atau dengan cleaning)
df['ingredients_clean'] = df['Ingredients Cleaned'].fillna('')

In [28]:

# Fitur numerik
num_features = ['Total Ingredients', 'Total Steps', 'Loves']
X_num = df[num_features].fillna(0).values

# Fitur teks (TF-IDF)
vectorizer = TfidfVectorizer(max_features=150, stop_words='english', min_df=2)
X_tfidf = vectorizer.fit_transform(df['ingredients_clean']).toarray()

# Gabungkan semua fitur (tanpa Category sebagai input, karena akan menjadi target)
X = np.hstack([X_num, X_tfidf])
print(f"Total fitur (X): {X.shape}")

# Target (Category, untuk klasifikasi)
# Gunakan LabelEncoder untuk mengubah kategori string menjadi angka integer
le_category = LabelEncoder()
y = le_category.fit_transform(df['Category'])
print(f"Target (y) shape: {y.shape}")
print(f"Mapping Kategori: {list(le_category.classes_)}")

Total fitur (X): (5000, 153)
Target (y) shape: (5000,)
Mapping Kategori: ['ayam', 'ikan', 'kambing', 'sapi', 'tahu', 'telur', 'tempe', 'udang']


In [29]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")

Train: (3500, 153), Validation: (750, 153), Test: (750, 153)


In [30]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling selesai. Contoh nilai pertama train:\n", X_train_scaled[0])

Scaling selesai. Contoh nilai pertama train:
 [ 0.54604003  2.51281668 -0.17940142 -0.80274228 -0.17148266  3.3589917
 -0.22218308 -0.53959967 -0.34408364 -0.29698465 -0.1620462  -0.1845378
 -0.31818316 -0.43774725 -0.13998067 -0.16685347 -0.21312348  1.2170006
  0.61135576 -0.17773574 -0.13967777 -0.33144904  0.16246382 -0.20619184
 -0.27227526 -0.19294989 -0.24890013 -0.49273655  0.17651757 -0.13330797
 -0.21093188 -0.13115943 -0.1401594  -0.14723935 -0.12732578  2.23395092
 -0.15749919  2.39595793  0.47450235  0.99578348 -0.24171199 -0.2941304
 -0.16657324 -0.13719127 -0.14760592 -0.33351939 -0.14444755 -0.56428844
 -0.22608897  3.19791013 -0.56196663 -0.2003439   3.47888034 -0.41978114
 -0.3642457  -0.17377628 -0.21223826 -0.20220851 -0.22271563  0.82945557
 -0.15716484 -0.18317536 -0.20359351 -0.48462895 -0.14584997 -0.1653296
 -0.25038451 -0.14086025 -0.3452114  -0.45314393 -0.1715163  -0.19633296
 -0.22102194 -0.16193559 -0.1990072  -0.16988387 -0.47480244 -0.20781882
 -0.497584

In [31]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable

# 1. Definisikan Custom Layer dari model_mlp.py
@register_keras_serializable()
class IngredientsImportanceLayer(Layer):
    def __init__(self, factor=1.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs):
        return inputs * self.factor

# 2. Definisikan Custom Loss Function (Tidak digunakan secara langsung di compile)
@register_keras_serializable()
def custom_recipe_loss(y_true, y_pred):
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(loss)

# 3. Build Model (Sesuai dengan fungsi build_mlp_model)
input_dim = X_train_scaled.shape[1]  # Dinamis menyesuaikan 503 fitur
num_classes = len(le_category.classes_) # Dinamis menyesuaikan 8 kategori

input_layer = Input(shape=(input_dim,), name='input')
x = IngredientsImportanceLayer()(input_layer)

x = Dense(128, activation='relu')(x)
x = Dropout(0.4)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

output_layer = Dense(num_classes, activation='softmax', name='output')(x)

# Buat models
model = Model(inputs=input_layer, outputs=output_layer, name='SayurKita_klasifikasi')
# Menggunakan 'sparse_categorical_crossentropy' bawaan Keras
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "SayurKita_klasifikasi"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 153)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ingredients_importance_layer    │ (None, 153)            │             0 │
│ (IngredientsImportanceLayer)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,488 (111.28 KB)

 Trainable params: 28,488 (111.28 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
import tensorflow as tf

class StopAtAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, target=0.90):
        super().__init__()
        self.target = target
    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = logs.get('val_accuracy')
        if val_accuracy and val_accuracy >= self.target: # Periksa apakah lebih besar atau sama dengan target
            print(f"\n Target Akurasi {self.target} tercapai di epoch {epoch+1}. Stop training.")
            self.model.stop_training = True

In [33]:
import tensorflow as tf

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[StopAtAccuracy(target=0.90), tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_accuracy', mode='max')], # Mengubah callback dan monitor/mode EarlyStopping
    verbose=1
)


Epoch 1/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.2517 - loss: 2.0984 - val_accuracy: 0.5387 - val_loss: 1.5088
Epoch 2/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.4974 - loss: 1.4479 - val_accuracy: 0.7253 - val_loss: 1.0432
Epoch 3/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6397 - loss: 1.0665 - val_accuracy: 0.7920 - val_loss: 0.7335
Epoch 4/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7146 - loss: 0.8527 - val_accuracy: 0.8360 - val_loss: 0.5794
Epoch 5/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7851 - loss: 0.6549 - val_accuracy: 0.8600 - val_loss: 0.4865
Epoch 6/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8180 - loss: 0.5529 - val_accuracy: 0.8733 - val_loss: 0.4307
Epoch 7/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8286 - loss: 0.5001 - val_accuracy: 0.8827 - val_loss: 0.3963
Epoch 8/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8580 - loss: 0.4375 - val_accuracy: 0.8933

In [34]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
if test_accuracy >= 0.85: # Sesuaikan target untuk akurasi
    print("Target Akurasi \u2265 0.85 tercapai pada test set!")
else:
    print(f"Akurasi masih {test_accuracy:.4f}, perlu perbaikan.")


Test Accuracy: 0.9027
Target Akurasi ≥ 0.85 tercapai pada test set!


In [35]:
\
# Buat nama file dengan timestamp untuk menghindari overwrite
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
main_classifier_model_save_path = f"./SayurKita_klasifikasi.keras" # Tambahkan ekstensi .keras

# Simpan model
tf.keras.models.save_model(model, main_classifier_model_save_path)
print(f"Model berhasil dissimpan di: {main_classifier_model_save_path}")


Model berhasil disimpan di: ./SayurKita_klasifikasi.keras


In [36]:
import tensorflow as tf
import numpy as np

# Pastikan model_save_path sudah didefinisikan atau gunakan variabel yang sudah ada
# Diasumsikan `main_classifier_model_save_path` sudah ada dari proses penyimpanan model.
# Perbaikan: Menggunakan nama file model yang terakhir seharusnya disimpan.
model_save_path = './SayurKita_klasifikasi.keras'

# Load the saved model
loaded_model = tf.keras.models.load_model(model_save_path)

# Create a dummy input for demonstration (e.g., using the first test sample)
# Pastikan X_test_scaled, y_test, dan le_category sudah ada dari proses preprocessing
dummy_input = X_test_scaled[0].reshape(1, -1) # Reshape for a single sample

# Make a prediction
predictions = loaded_model.predict(dummy_input)
predicted_class_index = np.argmax(predictions, axis=1)[0]
predicted_category = le_category.inverse_transform([predicted_class_index])[0]

print(f"Dummy input shape: {dummy_input.shape}")
print(f"Prediction probabilities: {predictions[0]}")
print(f"Predicted class index: {predicted_class_index}")
print(f"Predicted category: {predicted_category}")

# You can compare this with the actual category for the first test sample
actual_class_index = y_test[0]
actual_category = le_category.inverse_transform([actual_class_index])[0]
print(f"Actual category for the first test sample: {actual_category}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
Dummy input shape: (1, 153)
Prediction probabilities: [5.5699762e-05 1.6158292e-04 4.2825457e-07 2.1649178e-07 1.5581027e-05
 7.1234623e-05 9.9967337e-01 2.1912796e-05]
Predicted class index: 6
Predicted category: tempe
Actual category for the first test sample: tempe
